In [ ]:
input_file = "/home/symmetry/facts/fables/Drones/rekon10/attachments/logs/rekon10-first-flight_1980-01-12_14-29-50.bin"
debug = False

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from pathlib import Path
from pymavlink import mavutil
from IPython.display import display as ipy_display, Image as ipy_image
import io, json, datetime, os

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    ipy_display(ipy_image(data=buf.getvalue()))

LOG = Path(input_file)
print(f"log: {LOG.name}  ({LOG.stat().st_size/1024:.0f} KB)")

In [ ]:
# parse_log() canonical: coordinator/analysis/ardupilot_log.py
WANT = {'VIBE','GPS','ATT','BARO','BAT','ESC','CTUN','MODE','ARM','EV','IMU',
        'MOTB','RCOU','MSG','ERR','XKF1','RATE','PARM'}
rows = {t: [] for t in WANT}

mlog = mavutil.mavlink_connection(str(LOG), robust_parsing=True)
t_min = None
while True:
    msg = mlog.recv_match(blocking=False)
    if msg is None: break
    mtype = msg.get_type()
    if mtype not in WANT: continue
    t = getattr(msg, 'TimeUS', None)
    if t is None: continue
    if t_min is None: t_min = t
    d = {'t_s': (t - t_min) / 1e6}
    for f in msg._fieldnames:
        d[f] = getattr(msg, f)
    rows[mtype].append(d)

F = {t: pd.DataFrame(rows[t]) for t in WANT if rows[t]}
duration = max(df['t_s'].max() for df in F.values())

# parameters logged at boot
parms = {r['Name']: r['Value'] for _, r in F['PARM'].iterrows()} if 'PARM' in F else {}

print(f"duration: {duration:.1f}s ({duration/60:.1f} min)")
print(f"firmware: {next((r.Message for _,r in F['MSG'].iterrows() if 'ArduCopter' in str(r.Message)), 'unknown')}")
print(f"frame:    {next((r.Message for _,r in F['MSG'].iterrows() if 'Frame' in str(r.Message)), 'unknown')}")
print(f"LOG_BITMASK: {int(parms.get('LOG_BITMASK', -1))}")

In [ ]:
# --- ASSUMPTIONS: fail loudly before trusting any analysis ---
issues = []

# GPS quality
gps = F.get('GPS', pd.DataFrame())
gps_status_min = gps['Status'].min() if not gps.empty else 0
gps_status_max = gps['Status'].max() if not gps.empty else 0
if gps_status_max < 3:
    issues.append(f"GPS never reached 3D fix (max Status={gps_status_max})")

# Vibration
vibe = F.get('VIBE', pd.DataFrame())
if not vibe.empty:
    vibe0 = vibe[vibe['IMU'] == 0]
    vibe_max = vibe0[['VibeX','VibeY','VibeZ']].max().max()
    clip_total = vibe0['Clip'].max()
    if vibe_max > 30:
        issues.append(f"Vibration exceeds 30 m/s^2 threshold (max={vibe_max:.1f})")
    if clip_total > 0:
        issues.append(f"IMU clipping detected (Clip={clip_total})")

# Log completeness: check for clean disarm at end
arm_df = F.get('ARM', pd.DataFrame())
if arm_df.empty:
    issues.append("No ARM records found -- log may be incomplete")
else:
    last_arm = arm_df.sort_values('t_s').iloc[-1]
    if last_arm['ArmState'] == 1:
        issues.append("Log ends while still armed -- possible crash or power loss")

# EKF errors during flight
err_df = F.get('ERR', pd.DataFrame())
ekf_errors = err_df[err_df['Subsys'] == 24] if not err_df.empty else pd.DataFrame()
ekf_error_count = len(ekf_errors[ekf_errors['ECode'] != 0]) if not ekf_errors.empty else 0

print("=== ASSUMPTIONS ===")
if issues:
    for i in issues:
        print(f"  WARNING: {i}")
else:
    print("  all ok")
print(f"  GPS status range: {gps_status_min}-{gps_status_max}  (3=3D fix, 5=RTK float, 6=RTK fixed)")
print(f"  Vibration max (IMU0): {vibe_max:.3f} m/s^2  clip={clip_total}")
print(f"  EKF variance errors: {ekf_error_count}")
if ekf_error_count > 0:
    print("  NOTE: EKF variance errors present -- attitude/position data may be unreliable during affected period")

In [ ]:
MODE_MAP = {0:'Stabilize',2:'AltHold',3:'Auto',4:'Guided',5:'Loiter',
            6:'RTL',9:'Land',16:'PosHold',17:'Brake',18:'Throw',21:'AutoTune'}
EV_MAP = {10:'armed',11:'disarmed',15:'auto_armed',16:'land_complete',
           18:'land_complete_maybe',25:'set_home',57:'arm_disallowed'}
SUBSYS_MAP = {2:'radio',3:'compass',5:'radio_fs',8:'gps',12:'ins',
              24:'ekf_var',25:'viso',26:'terrain',27:'nav',30:'failsafe'}

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
fig.suptitle(f"Flight overview -- {LOG.stem}", fontsize=10, fontweight='bold')

# Panel 1: altitude
axes[0].plot(F['BARO']['t_s'], F['BARO']['Alt'], color='steelblue', lw=1)
axes[0].set_ylabel('Rel Alt (m)')
axes[0].grid(True, alpha=0.3)

# Panel 2: attitude
att = F['ATT']
axes[1].plot(att['t_s'], att['Roll'], label='Roll', color='royalblue', lw=1)
axes[1].plot(att['t_s'], att['Pitch'], label='Pitch', color='darkorange', lw=1)
axes[1].set_ylabel('Angle (deg)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Panel 3: battery
bat = F['BAT']
ax2 = axes[2].twinx()
axes[2].plot(bat['t_s'], bat['Volt'], color='green', lw=1.2, label='Voltage')
ax2.plot(bat['t_s'], bat['Curr'], color='crimson', lw=1, alpha=0.7, label='Current')
axes[2].set_ylabel('Voltage (V)', color='green')
ax2.set_ylabel('Current (A)', color='crimson')
axes[2].grid(True, alpha=0.3)

# Panel 4: motor outputs
rcou = F['RCOU']
for ch, col in zip(['C1','C2','C3','C4'], ['#e41a1c','#377eb8','#4daf4a','#984ea3']):
    axes[3].plot(rcou['t_s'], rcou[ch], lw=1, alpha=0.8, label=ch, color=col)
axes[3].set_ylabel('PWM (us)')
axes[3].set_xlabel('Time (s)')
axes[3].legend(fontsize=8, ncol=4)
axes[3].grid(True, alpha=0.3)

# Arm/disarm verticals (only if ARM records exist)
arm_df = F.get('ARM', pd.DataFrame())
if not arm_df.empty:
    arm_df = arm_df.sort_values('t_s')
    for _, row in arm_df.iterrows():
        color = 'green' if row['ArmState'] else 'red'
        for ax in axes:
            ax.axvline(row['t_s'], color=color, lw=1.2, linestyle='--', alpha=0.6)

# ERR events on altitude panel
err_df = F.get('ERR', pd.DataFrame())
for _, row in err_df.iterrows():
    if row['ECode'] != 0:
        axes[0].axvline(row['t_s'], color='red', lw=0.8, linestyle=':', alpha=0.8)
        axes[0].text(row['t_s']+0.3, 0.95, SUBSYS_MAP.get(int(row['Subsys']), f"E{int(row['Subsys'])}"),
                     fontsize=6, rotation=90, color='red', va='top', transform=axes[0].get_xaxis_transform())

# Mode change markers
for _, row in F.get('MODE', pd.DataFrame()).iterrows():
    axes[0].axvline(row['t_s'], color='purple', lw=0.8, linestyle=':', alpha=0.5)
    axes[0].text(row['t_s']+0.3, 0.05, MODE_MAP.get(int(row['Mode']), f"M{int(row['Mode'])}"),
                 fontsize=6, rotation=90, color='purple', va='bottom', transform=axes[0].get_xaxis_transform())

plt.tight_layout()
show(fig)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle('Vibration (VIBE)', fontsize=10, fontweight='bold')

for imu_id, ax in zip([0, 1], axes):
    v = vibe[vibe['IMU'] == imu_id]
    if v.empty: continue
    ax.plot(v['t_s'], v['VibeX'], label='X', color='#e41a1c', lw=1)
    ax.plot(v['t_s'], v['VibeY'], label='Y', color='#377eb8', lw=1)
    ax.plot(v['t_s'], v['VibeZ'], label='Z', color='#4daf4a', lw=1)
    ax.axhline(30, color='orange', lw=1, linestyle='--', label='30 m/s^2 warn')
    ax.axhline(60, color='red', lw=1, linestyle='--', label='60 m/s^2 bad')
    ax.set_ylabel(f'IMU{imu_id} (m/s^2)')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8, ncol=5)
    ax.grid(True, alpha=0.3)
    ax2 = ax.twinx()
    ax2.plot(v['t_s'], v['Clip'], color='purple', lw=1, alpha=0.5)
    ax2.set_ylabel('Clip count', color='purple')

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
show(fig)

for imu_id in [0, 1]:
    v = vibe[vibe['IMU'] == imu_id]
    if v.empty: continue
    print(f"IMU{imu_id}  VibeX max={v['VibeX'].max():.3f}  VibeY max={v['VibeY'].max():.3f}  "
          f"VibeZ max={v['VibeZ'].max():.3f}  Clip={v['Clip'].max():.0f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
fig.suptitle('GPS quality', fontsize=10, fontweight='bold')

gps0 = gps[gps['I'] == 0]

axes[0].plot(gps0['t_s'], gps0['Status'], color='steelblue', lw=1.2, drawstyle='steps-post')
axes[0].set_ylabel('GPS Status')
axes[0].set_yticks([0,1,2,3,4,5,6])
axes[0].set_yticklabels(['No GPS','No fix','2D fix','3D fix','DGPS','RTK float','RTK fixed'], fontsize=7)
axes[0].axhline(5, color='green', lw=0.8, linestyle='--', alpha=0.5, label='RTK float min')
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

axes[1].plot(gps0['t_s'], gps0['NSats'], color='darkorange', lw=1.2)
axes[1].set_ylabel('Satellites')
axes[1].grid(True, alpha=0.3)

axes[2].plot(gps0['t_s'], gps0['HDop'], color='crimson', lw=1.2)
axes[2].axhline(2.0, color='orange', lw=0.8, linestyle='--', alpha=0.7, label='HDop 2.0')
axes[2].set_ylabel('HDop')
axes[2].set_xlabel('Time (s)')
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
show(fig)

print(f"GPS Status: min={gps0['Status'].min()}  max={gps0['Status'].max()}  "
      f"(3D fix={gps0['Status'].value_counts().get(3,0)} samples)")
print(f"NSats: min={gps0['NSats'].min():.0f}  max={gps0['NSats'].max():.0f}  mean={gps0['NSats'].mean():.1f}")
print(f"HDop:  min={gps0['HDop'].min():.2f}  max={gps0['HDop'].max():.2f}  mean={gps0['HDop'].mean():.2f}")
print("NOTE: GPS never reached RTK float (Status=5) -- position quality limited to 3D fix")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
fig.suptitle('Battery', fontsize=10, fontweight='bold')

bat0 = bat[bat['Inst'] == 0]

axes[0].plot(bat0['t_s'], bat0['Volt'], color='green', lw=1.2, label='Measured')
axes[0].plot(bat0['t_s'], bat0['VoltR'], color='lime', lw=1, linestyle='--', label='Resting est.')
axes[0].set_ylabel('Voltage (V)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(bat0['t_s'], bat0['Curr'], color='crimson', lw=1)
axes[1].set_ylabel('Current (A)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(bat0['t_s'], bat0['RemPct'], color='steelblue', lw=1.2)
axes[2].set_ylabel('Remaining (%)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylim(0, 105)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
show(fig)

print(f"Voltage: start={bat0['Volt'].iloc[0]:.2f}V  end={bat0['Volt'].iloc[-1]:.2f}V")
print(f"Current: max={bat0['Curr'].max():.1f}A  mean={bat0['Curr'].mean():.1f}A")
print(f"Consumed: {bat0['CurrTot'].iloc[-1] - bat0['CurrTot'].iloc[0]:.0f} mAh")
print(f"Remaining: {bat0['RemPct'].iloc[0]:.0f}% -> {bat0['RemPct'].iloc[-1]:.0f}%")

In [ ]:
att = F['ATT']
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Attitude tracking (desired vs actual)', fontsize=10, fontweight='bold')

for ax, des, actual, label, color in [
    (axes[0], 'DesRoll', 'Roll', 'Roll (deg)', '#e41a1c'),
    (axes[1], 'DesPitch', 'Pitch', 'Pitch (deg)', '#377eb8'),
    (axes[2], 'DesYaw', 'Yaw', 'Yaw (deg)', '#4daf4a'),
]:
    ax.plot(att['t_s'], att[des], lw=1, linestyle='--', color=color, alpha=0.6, label='desired')
    ax.plot(att['t_s'], att[actual], lw=1, color=color, label='actual')
    ax.set_ylabel(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

_arm_df = F.get('ARM', pd.DataFrame())
if not _arm_df.empty:
    _arm_df = _arm_df.sort_values('t_s')
armed_at = _arm_df[_arm_df['ArmState'] == 1]['t_s'].values if not _arm_df.empty else []
disarmed_at = _arm_df[_arm_df['ArmState'] == 0]['t_s'].values if not _arm_df.empty else []
for t_arm in armed_at:
    t_disarm = disarmed_at[disarmed_at > t_arm][0] if any(disarmed_at > t_arm) else duration
    for ax in axes:
        ax.axvspan(t_arm, t_disarm, alpha=0.08, color='green')

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
show(fig)

roll_err = (att['Roll'] - att['DesRoll']).abs()
pitch_err = (att['Pitch'] - att['DesPitch']).abs()
print(f"Roll error:  mean={roll_err.mean():.2f}deg  max={roll_err.max():.2f}deg")
print(f"Pitch error: mean={pitch_err.mean():.2f}deg  max={pitch_err.max():.2f}deg")
print("NOTE: large errors are pre-arm (disarmed, motors off) -- filter to armed interval for tuning assessment")

In [ ]:
ekf = F.get('XKF1', pd.DataFrame())

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle('EKF and error events', fontsize=10, fontweight='bold')

ekf0 = ekf[ekf['C'] == 0]
axes[0].plot(ekf0['t_s'], ekf0['VN'], label='VN', lw=1)
axes[0].plot(ekf0['t_s'], ekf0['VE'], label='VE', lw=1)
axes[0].plot(ekf0['t_s'], ekf0['VD'], label='VD', lw=1)
axes[0].set_ylabel('EKF NED velocity (m/s)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

ax = axes[1]
err_active = err_df[err_df['ECode'] != 0] if not err_df.empty and 'ECode' in err_df.columns else pd.DataFrame()
for _, row in err_active.iterrows():
    label = SUBSYS_MAP.get(int(row['Subsys']), f"sub{int(row['Subsys'])}")
    ax.axvline(row['t_s'], color='red', lw=2, alpha=0.7)
    ax.text(row['t_s']+0.2, 0.5, label, fontsize=7, transform=ax.get_xaxis_transform(), color='red')
ax.set_ylabel('Errors')
ax.set_xlabel('Time (s)')
ax.set_yticks([])
ax.grid(True, alpha=0.3)

key_msgs = ['EKF3', 'Glitch', 'PreArm', 'variance']
for _, row in F['MSG'].iterrows():
    if any(k in str(row['Message']) for k in key_msgs):
        axes[0].axvline(row['t_s'], color='orange', lw=0.8, linestyle=':', alpha=0.7)

plt.tight_layout()
show(fig)

print("Key messages:")
for _, row in F['MSG'].iterrows():
    if any(k in str(row['Message']) for k in key_msgs + ['yaw', 'lane']):
        print(f"  t+{row['t_s']:.1f}s  {row['Message']}")

In [ ]:
esc = F.get('ESC', pd.DataFrame())
n_escs = int(esc['Instance'].max()) + 1 if not esc.empty else 0

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
fig.suptitle(f'ESC data ({n_escs} motors)', fontsize=10, fontweight='bold')

colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3']
for i in range(n_escs):
    e = esc[esc['Instance'] == i]
    axes[0].plot(e['t_s'], e['RPM'], label=f'M{i+1}', lw=1, color=colors[i % 4])
    axes[1].plot(e['t_s'], e['Curr'], label=f'M{i+1}', lw=1, color=colors[i % 4])
    axes[2].plot(e['t_s'], e['Temp'], label=f'M{i+1}', lw=1, color=colors[i % 4])

axes[0].set_ylabel('RPM')
axes[0].legend(fontsize=8, ncol=4)
axes[0].grid(True, alpha=0.3)
axes[1].set_ylabel('Current (A)')
axes[1].legend(fontsize=8, ncol=4)
axes[1].grid(True, alpha=0.3)
axes[2].set_ylabel('Temp (C)')
axes[2].set_xlabel('Time (s)')
axes[2].legend(fontsize=8, ncol=4)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
show(fig)

if not esc.empty:
    print("ESC summary:")
    for i in range(n_escs):
        e = esc[esc['Instance'] == i]
        print(f"  M{i+1}: RPM max={e['RPM'].max():.0f}  Curr max={e['Curr'].max():.1f}A  "
              f"Temp max={e['Temp'].max():.1f}C  Err={e['Err'].max():.0f}")

In [ ]:
print("=" * 60)
print("FLIGHT SUMMARY")
print("=" * 60)
print(f"Log:        {LOG.name}")
print(f"Duration:   {duration:.1f}s ({duration/60:.1f} min)")
print(f"Firmware:   {next((r.Message for _,r in F['MSG'].iterrows() if 'ArduCopter' in str(r.Message)), '?')}")
print()

total_armed = sum(
    (disarmed_at[disarmed_at > t][0] if any(disarmed_at > t) else duration) - t
    for t in armed_at
) if len(armed_at) else 0
print(f"Armed time: {total_armed:.1f}s across {len(armed_at)} arm event(s)")
print()
print("Observations:")

vibe_status = f"OVER THRESHOLD: max {vibe_max:.1f} m/s^2, Clip={clip_total:.0f}" if vibe_max > 30 else f"ok: max {vibe_max:.1f} m/s^2"
print(f"  Vibration:  {vibe_status}")
_gps_label = {3:'3D fix',4:'DGPS',5:'RTK float',6:'RTK fixed'}.get(gps_status_max, f'status {gps_status_max}')
print(f"  GPS:        max {_gps_label} (status {gps_status_max}), NSats {gps0['NSats'].mean():.0f} avg, HDop {gps0['HDop'].mean():.2f} avg")
print(f"  Battery:    {bat0['Volt'].iloc[0]:.1f}V start -> {bat0['Volt'].iloc[-1]:.1f}V end, "
      f"{bat0['RemPct'].iloc[0]:.0f}% -> {bat0['RemPct'].iloc[-1]:.0f}% remaining")
if ekf_error_count:
    print(f"  EKF:        {ekf_error_count} variance error(s) -- lane switch + compass anomaly caused disarm at t+71s")
    print(f"              mag xy diff 265 > 100 threshold -- compass interference, investigate mounting/sources")
modes_seen = [MODE_MAP.get(int(r.Mode), f"M{int(r.Mode)}") for _,r in F.get('MODE', pd.DataFrame()).iterrows()]
if modes_seen:
    print(f"  Modes:      {' -> '.join(modes_seen)}")

# AutoTune summary from MSG records
autotune_msgs = [r.Message for _,r in F['MSG'].iterrows()
                 if any(k in str(r.Message) for k in ['AutoTune: Pitch','AutoTune: Roll','AutoTune: Yaw','AutoTune: Success','AutoTune: Stopped'])]
if autotune_msgs:
    print()
    print("AutoTune activity:")
    for m in autotune_msgs[-12:]:
        print(f"  {m}")

if issues:
    print()
    print("Assumption violations (analysis results unreliable):")
    for i in issues:
        print(f"  ! {i}")

In [ ]:
out_dir = Path(os.environ.get('NB_OUTPUT_DIR', str(LOG.parent)))
manifest = {
    'notebook': 'flight-analysis.ipynb',
    'input_file': str(LOG),
    'run_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'duration_s': float(duration),
    'armed_s': float(total_armed),
    'gps_status_max': int(gps_status_max),
    'vibe_max_ms2': float(vibe_max),
    'clip_total': int(clip_total),
    'ekf_errors': int(ekf_error_count),
    'assumption_warnings': issues,
    'status': 'ok' if not issues else 'warnings',
    'debug': debug,
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(f"manifest -> {out_dir / 'manifest.json'}")
print(json.dumps(manifest, indent=2))

In [ ]:
# AGENT-READABLE STATS -- consolidated JSON for programmatic consumption
def _s(series):
    s = series.dropna()
    if s.empty: return {}
    return {'min': round(float(s.min()),3), 'max': round(float(s.max()),3),
            'mean': round(float(s.mean()),3), 'p95': round(float(s.quantile(0.95)),3)}

_armed_mask = lambda df: df[(df['t_s'] >= armed_at[0]) & (df['t_s'] <= (disarmed_at[disarmed_at > armed_at[0]][0] if any(disarmed_at > armed_at[0]) else duration))] if len(armed_at) else df.iloc[0:0]
_att_armed = _armed_mask(att)

agent_stats = {
    'log': LOG.name,
    'duration_s': round(duration, 2),
    'armed_s': round(sum(
        (disarmed_at[disarmed_at > t][0] if any(disarmed_at > t) else duration) - t
        for t in armed_at
    ) if len(armed_at) else 0, 2),
    'arm_events': [{'t_s': round(r.t_s,2), 'state': int(r.ArmState), 'method': int(r.Method)}
                   for _,r in (_arm_df.sort_values('t_s').iterrows() if not _arm_df.empty else [])],
    'mode_events': [{'t_s': round(r.t_s,2), 'mode': MODE_MAP.get(int(r.Mode), int(r.Mode))}
                    for _,r in (F['MODE'].sort_values('t_s').iterrows() if 'MODE' in F else [])],
    'vibe': {
        f'imu{i}': {
            'x': _s(vibe[vibe['IMU']==i]['VibeX']),
            'y': _s(vibe[vibe['IMU']==i]['VibeY']),
            'z': _s(vibe[vibe['IMU']==i]['VibeZ']),
            'clip_max': float(vibe[vibe['IMU']==i]['Clip'].max()),
            'pct_over_30': round(float((vibe[vibe['IMU']==i][['VibeX','VibeY','VibeZ']].max(axis=1) > 30).mean()*100),1),
        } for i in [0,1] if not vibe[vibe['IMU']==i].empty
    },
    'gps': {
        'status_counts': {str(k): int(v) for k,v in gps0['Status'].value_counts().items()},
        'nsats': _s(gps0['NSats']), 'hdop': _s(gps0['HDop']),
        'lat_range': [round(float(gps0['Lat'].min()),6), round(float(gps0['Lat'].max()),6)],
        'lng_range': [round(float(gps0['Lng'].min()),6), round(float(gps0['Lng'].max()),6)],
    },
    'battery': {
        'volt': {'start': round(float(bat0['Volt'].iloc[0]),2), 'end': round(float(bat0['Volt'].iloc[-1]),2),
                 **_s(bat0['Volt'])},
        'curr': _s(bat0['Curr']),
        'consumed_mah': round(float(bat0['CurrTot'].iloc[-1] - bat0['CurrTot'].iloc[0]),0),
        'rem_pct': {'start': int(bat0['RemPct'].iloc[0]), 'end': int(bat0['RemPct'].iloc[-1])},
    },
    'attitude_armed': {
        'roll_err': _s((_att_armed['Roll'] - _att_armed['DesRoll']).abs()),
        'pitch_err': _s((_att_armed['Pitch'] - _att_armed['DesPitch']).abs()),
    } if not _att_armed.empty else 'no_armed_data',
    'ekf_errors': [{'t_s': round(r.t_s,2), 'subsys': SUBSYS_MAP.get(int(r.Subsys), int(r.Subsys)), 'code': int(r.ECode)}
                   for _,r in (err_df[err_df['ECode']!=0] if not err_df.empty and 'ECode' in err_df.columns else pd.DataFrame()).iterrows()],
    'esc': {
        f'm{i+1}': {'rpm': _s(esc[esc['Instance']==i]['RPM']),
                    'curr': _s(esc[esc['Instance']==i]['Curr']),
                    'temp': _s(esc[esc['Instance']==i]['Temp']),
                    'err_max': float(esc[esc['Instance']==i]['Err'].max())}
        for i in range(n_escs)
    } if not esc.empty else {},
    'key_messages': [{'t_s': round(r.t_s,2), 'msg': str(r.Message)} for _,r in F['MSG'].iterrows()
                     if any(k in str(r.Message) for k in ['EKF','Glitch','PreArm','variance','yaw','lane','Compass','error'])],
    'assumption_warnings': issues,
    'status': 'ok' if not issues else 'warnings',
}

print(json.dumps(agent_stats, indent=2))